In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ (her notebook'un ilk hücresi) ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "main"                                         # kendi branch'inizde çalışıyorsanız değiştirin
try:
    TOKEN = userdata.get("GH_TOKEN")                        # sadece private repo için gerekir
except userdata.SecretNotFoundError:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .
if "/content/repo/src" not in sys.path:      # editable install is only seen after a kernel restart
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()          # config/config.yaml + DATA_ROOT
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 10 — Preliminary kNN prototype and service interface

Stage 1.10. This notebook contains no business logic; it only calls `recsys` modules.

* Inputs: stages **`splits`** (1.9: grouped split, train-boundary targets, fitted pipeline,
  catalogue ingredient matrix, development sample) and **`loaded`** (raw minutes for display).
* Output: ONE atomic **`knn_prototype`** version: the fitted `KNNRecipeRecommender` (the artefact
  the Phase 4 API will load) plus the validation tables, each validated against
  `KNN_PROTOTYPE_OUTPUT_SCHEMA`.
* Ingredient distance is **Jaccard** (config `knn_prototype.metric`); Euclidean is never used
  (4.5). k and the metric are untuned here: tuning on validation is P2.
* **The test set is not touched.** Every evaluation call names its split and
  `recsys.evaluation.metrics` refuses `"test"` (the override stays locked until P2).

In [ ]:
from recsys import registry
from recsys.contracts import KNN_PROTOTYPE_OUTPUT_SCHEMA, LOADED_INTERACTIONS_SCHEMA, validate
from recsys.data.split import apply_assignments
from recsys.evaluation.metrics import (
    HeldOutRecipes,
    TestSetAccessError,
    check_split,
    compare_classifiers,
    compare_regressors,
)
from recsys.io import ArtifactStore
from recsys.models.knn import (
    KNNRecipeRecommender,
    MajorityBaseline,
    MeanBaseline,
    RatingClassifierKNN,
    RatingRegressorKNN,
    catalogue_table,
    example_recommendations,
    query_vectorizer_from_pipeline,
    recipe_rating_dataset,
)

KCFG = CFG.section("knn_prototype")
LABELS = CFG.section("binning")["avg_rating"]["labels"]

store = ArtifactStore.from_config(CFG)
splits = store.load("splits")
loaded = store.load("loaded")
interactions = validate(loaded["interactions"], LOADED_INTERACTIONS_SCHEMA).reset_index(drop=True)
grouped = apply_assignments(interactions, splits["grouped_assignments"], "group_stratified")
pipeline = splits["pipeline"]
matrix = splits["ingredient_matrix"]
print("inputs: splits", splits.version, "| loaded", loaded.version,
      "| ingredient matrix", matrix.shape, "| train interactions", len(grouped.train))

## 1. Recommender over the whole catalogue

`KNNRecipeRecommender` indexes **every** catalogue recipe, rated or not. Known ratings are the
average of the **train** interactions only; any other recipe (val / test recipes, recipes never
reviewed) gets a rating estimated from its `rating_neighbors` most similar rated recipes
(distance-weighted mean), flagged `rating_is_estimated=True`. Queries are vectorised with the
normaliser and vocabulary of the fitted 1.9 pipeline, so query and catalogue share one vocab.

In [ ]:
metric = registry.create("distance", KCFG["metric"])
catalogue = catalogue_table(splits["recipe_features"], loaded["recipes"])
recommender = KNNRecipeRecommender(
    vectorizer=query_vectorizer_from_pipeline(pipeline),
    metric=metric,
    rating_neighbors=KCFG["rating_neighbors"],
    batch_size=KCFG["batch_size"],
).fit(catalogue, grouped.train, ingredient_matrix=matrix)
print("catalogue recipes", len(catalogue), "| with a known (train) rating", recommender.n_rated_)

## 2. Example queries

Three queries from `config.yaml: knn_prototype.example_queries`; the third uses the
`minutes_bin` filter. `unknown_ingredients` lists query terms that are not in the vocabulary
(the app shows them as "not recognised").

In [ ]:
examples = example_recommendations(recommender, KCFG["example_queries"], KCFG["top_k"])
examples[["query_id", "query", "filters", "rank", "name", "similarity", "predicted_rating",
          "rating_is_estimated", "minutes", "minutes_bin", "unknown_ingredients"]]

**Finding (draft):** *Fill in from the table above: are the top results plausible, how many
ratings are estimated, does the filter keep only `quick` recipes? Jaccard divides by the union,
so recipes with few ingredients can rank high on a single shared ingredient.*

## 3. Rating models on the VALIDATION set (development sample)

Recipe-level task: predict the rating class (`low` / `medium` / `high`, train boundaries from 1.9)
of a validation recipe from the classes of its most similar train recipes. Only recipes of the
1.9 development sample are used, and the grouped split guarantees that no validation recipe has a
train interaction.

* `majority`: always the most frequent train class (baseline only, 4.4).
* `knn_uniform` / `knn_distance`: k = `knn_prototype.n_neighbors`, one vote each vs vote
  `1 / (d + eps)`.
* Regression on `avg_rating` (MAE / RMSE) against the train mean.

In [ ]:
dev_ids = splits["dev_recipes"]["id"]
catalogue_ids = catalogue["id"]
X_tr, y_tr, _ = recipe_rating_dataset(matrix, catalogue_ids, splits["targets_train"], dev_ids)
X_val, y_val, val_ids = recipe_rating_dataset(matrix, catalogue_ids, splits["targets_val"], dev_ids)
held_out = HeldOutRecipes.from_targets(splits["targets_test"])  # data check of the test-set guard
knn_params = {"metric": metric, "n_neighbors": KCFG["n_neighbors"],
              "batch_size": KCFG["batch_size"], "eps": KCFG["distance_eps"]}
classification, confusion = compare_classifiers(
    {
        "majority": MajorityBaseline(),
        "knn_uniform": RatingClassifierKNN(weights="uniform", **knn_params),
        "knn_distance": RatingClassifierKNN(weights="distance", **knn_params),
    },
    X_tr, y_tr, X_val, y_val, labels=LABELS, split="val",
    eval_recipe_ids=val_ids, held_out=held_out,
)
print("train recipes", X_tr.shape[0], "| validation recipes", X_val.shape[0])
classification

In [ ]:
confusion.pivot_table(index=["model", "true_label"], columns="pred_label", values="count")

In [ ]:
Xr_tr, yr_tr, _ = recipe_rating_dataset(
    matrix, catalogue_ids, splits["targets_train"], dev_ids, target_col="avg_rating")
Xr_val, yr_val, val_ids_r = recipe_rating_dataset(
    matrix, catalogue_ids, splits["targets_val"], dev_ids, target_col="avg_rating")
regression = compare_regressors(
    {
        "mean": MeanBaseline(),
        "knn_uniform": RatingRegressorKNN(weights="uniform", **knn_params),
        "knn_distance": RatingRegressorKNN(weights="distance", **knn_params),
    },
    Xr_tr, yr_tr, Xr_val, yr_val, split="val",
    eval_recipe_ids=val_ids_r, held_out=held_out,
)
regression

**Why accuracy is misleading (draft):** *Fill in from the tables above. The classes are
imbalanced (see 1.9 `class_balance`), so the majority baseline already reaches a high accuracy
while its balanced accuracy is 1 / n_classes and its macro-F1 is low; compare the kNN rows on
balanced accuracy and macro-F1, and read the confusion matrix for the minority classes.*

## 4. Test-set guard

Phase 1 never scores the test set. The guard below must raise.

In [ ]:
try:
    check_split("test")
    raise AssertionError("the test split was accepted")
except TestSetAccessError as err:
    print("refused as expected:", err)
try:
    check_split("test", allow_test=True)
    raise AssertionError("the locked override was accepted")
except TestSetAccessError as err:
    print("override locked as expected:", err)

## 5. Save stage `knn_prototype` (one atomic version)

The recommender and all tables go into ONE version; the tables are validated against
`KNN_PROTOTYPE_OUTPUT_SCHEMA` before anything is written. `KNNRecipeRecommender.load(store)` is how
the Phase 4 API will read it.

In [ ]:
saved = recommender.save(
    store,
    inputs=[splits, loaded],
    params={
        "knn_prototype": KCFG,
        "labels": LABELS,
        "evaluation_split": "val",
        "evaluation_recipes": "splits/dev_recipes",
        "n_train_recipes": int(X_tr.shape[0]),
        "n_val_recipes": int(X_val.shape[0]),
        "random_state": CFG.random_state,
    },
    extra_objects={
        "classification_metrics": classification,
        "regression_metrics": regression,
        "confusion_matrices": confusion,
        "example_recommendations": examples,
    },
    schemas=KNN_PROTOTYPE_OUTPUT_SCHEMA,
)
print("saved:", saved.stage, saved.version, "->", saved.path)
reloaded = KNNRecipeRecommender.load(store)  # sha256-verified
first = KCFG["example_queries"][0]["ingredients"]
assert reloaded.recommend(first, k=3) == recommender.recommend(first, k=3)
[r.to_dict() for r in reloaded.recommend(first, k=3)]